# Training & Testing Klasifikasi Ekspresi Wajah di Google Colab

**Persiapan (sekali saja)**
1. Di Google Drive buat folder `fer_project`, isi dengan:
   - `fer_app.zip` — folder proyek (**tanpa** `.venv` dan `data`)
   - `fer2013.zip` — zip dataset Kaggle (berisi folder `train` dan `test`)
2. Menu **Runtime → Change runtime type → T4 GPU**, lalu jalankan sel dari atas ke bawah.

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))   # harus tidak kosong

## 1. Ambil proyek & dataset dari Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.chdir("/content")

# Definisikan path folder di Google Drive sesuai namanya
DRIVE_DIR = "/content/drive/MyDrive/SEMESTER 3/fer project"

# Hapus folder lama jika ada, lalu buat ulang
!rm -rf /content/fer_app
!mkdir -p /content/fer_app/data

# Lakukan unzip file dari Google Drive dengan penanganan spasi yang benar
!unzip -q "$DRIVE_DIR/fer_app.zip" -d /content/fer_app
!unzip -q "$DRIVE_DIR/fer2013.zip" -d /content/fer_app/data

# Pindah ke folder fer_app
%cd /content/fer_app

In [ ]:
import os
from pathlib import Path

root = Path("/content/fer_app/data")
found = [p.parent for p in root.rglob("train") if p.is_dir() and (p.parent / "test").is_dir()]
assert found, "Folder train/test tidak ditemukan. Cek isi fer2013.zip"
os.environ["FER_DATA_DIR"] = str(found[0])        # dibaca oleh config.py
print("Dataset:", found[0])
for split in ("train", "test"):
    n = sum(1 for f in (found[0] / split).rglob("*") if f.is_file())
    print(f"{split}: {n} gambar")                 # sekitar 28709 dan 7178

## 2. Training (2 fase) + evaluasi otomatis

In [ ]:
!python -m src.training.train

## 3. Lihat hasil evaluasi

In [ ]:
import pandas as pd
from IPython.display import Image, display

print(open("reports/summary.json").read())
display(pd.read_csv("reports/per_class_metrics.csv", index_col=0).round(3))
display(pd.read_csv("reports/confused_pairs.csv").head(10))
display(Image("reports/confusion_matrix.png"))
display(Image("reports/training_history.png"))
display(Image("reports/confident_errors.png"))

## 4. Ekspor model ringan & simpan hasil
Checkpoint training menyimpan state optimizer sehingga ukurannya besar (bisa >100 MB, ditolak GitHub).
Sel ini membuat versi ringan khusus untuk deploy.

In [ ]:
from tensorflow import keras

model = keras.models.load_model("models/fer_vgg16.keras", compile=False)
os.makedirs("export/models", exist_ok=True)
model.save("export/models/fer_vgg16.keras")
!ls -lh models export/models

In [ ]:
!cp -r reports export/reports
!cd export && zip -qr hasil_training.zip models reports
!cp export/hasil_training.zip "$DRIVE_DIR/"        # cadangan di Drive

from google.colab import files
files.download("export/hasil_training.zip")         # unduh ke laptop

**Selanjutnya (di laptop/VS Code):** ekstrak `hasil_training.zip`, isi folder `models/` dan `reports/`
timpa ke folder `fer_app`, lalu `streamlit run app.py`.